In [1]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────
# 1. Build deep networks with configurable init
# ──────────────────────────────────────────────

def make_deep_net(n_layers: int, width: int,
                  activation: str = "sigmoid",
                  weight_scale: float = 0.01) -> nn.Sequential:
    """
    Build a deep network.
      - weight_scale=0.01  → small weights  → vanishing
      - weight_scale=2.0   → large weights  → exploding
    """
    act_map = {
        "sigmoid": nn.Sigmoid,
        "tanh":    nn.Tanh,
        "relu":    nn.ReLU,
    }
    layers = []
    for _ in range(n_layers):
        lin = nn.Linear(width, width)
        nn.init.normal_(lin.weight, std=weight_scale)
        nn.init.zeros_(lin.bias)
        layers.append(lin)
        layers.append(act_map[activation]())
    # Output layer
    out = nn.Linear(width, 1)
    nn.init.normal_(out.weight, std=weight_scale)
    layers.append(out)
    return nn.Sequential(*layers)


# ──────────────────────────────────────────────
# 2. Measure gradient + activation stats
# ──────────────────────────────────────────────

def measure_all(model: nn.Sequential, x: torch.Tensor,
                y: torch.Tensor) -> dict:
    """
    Forward + backward pass.
    Record activations (forward) and gradients (backward) per layer.
    """
    activations = []
    hooks = []

    # Hook to capture activations during forward pass
    for module in model:
        if isinstance(module, (nn.Sigmoid, nn.Tanh, nn.ReLU)):
            def hook_fn(mod, inp, out, store=activations):
                store.append(out.detach().clone())
            hooks.append(module.register_forward_hook(hook_fn))

    # Forward + backward
    model.zero_grad()
    pred = model(x)
    loss = nn.MSELoss()(pred, y)
    loss.backward()

    # Remove hooks
    for h in hooks:
        h.remove()

    # Collect gradient stats
    grad_stats = []
    for module in model:
        if isinstance(module, nn.Linear):
            g = module.weight.grad
            grad_stats.append({
                "grad_mean": g.abs().mean().item(),
                "grad_max":  g.abs().max().item(),
            })

    # Collect activation stats
    act_stats = []
    for a in activations:
        act_stats.append({
            "act_mean": a.abs().mean().item(),
            "act_max":  a.abs().max().item(),
            "act_std":  a.std().item(),
        })

    return {
        "grad_stats": grad_stats,
        "act_stats":  act_stats,
        "loss":       loss.item(),
        "pred_mean":  pred.abs().mean().item(),
    }


# ──────────────────────────────────────────────
# 3. Visualise gradient + activation flow
# ──────────────────────────────────────────────

def print_flow(stats: dict, label: str):
    """Print gradient and activation magnitudes with visual bars."""
    grad_stats = stats["grad_stats"]
    act_stats  = stats["act_stats"]

    # ── Gradients ──
    print(f"  Gradients (weight.grad):")
    print(f"  {'Layer':<8} {'Mean |grad|':<16} Bar")
    max_g = max(s["grad_mean"] for s in grad_stats) + 1e-30

    for i, s in enumerate(grad_stats):
        bar_len = int(35 * s["grad_mean"] / max_g)
        bar = "█" * bar_len + "░" * (35 - bar_len)
        print(f"  {i:<8} {s['grad_mean']:<16.2e} {bar}")

    ratio = grad_stats[0]["grad_mean"] / (grad_stats[-1]["grad_mean"] + 1e-30)
    print(f"  Ratio (layer 0 / layer {len(grad_stats)-1}): {ratio:.2e}")

    if ratio < 1e-3:
        print("  ⚠ VANISHING\n")
    elif ratio > 1e3:
        print("  ⚠ EXPLODING\n")
    else:
        print("  ✓ Healthy\n")

    # ── Activations ──
    if act_stats:
        print(f"  Activations (after each activation fn):")
        print(f"  {'Layer':<8} {'Mean |act|':<14} {'Std':<14}")
        for i, s in enumerate(act_stats):
            print(f"  {i:<8} {s['act_mean']:<14.4f} {s['act_std']:<14.4f}")
        print()


# ──────────────────────────────────────────────
# 4. Scenario A: VANISHING gradients (sigmoid)
# ──────────────────────────────────────────────

def demo_vanishing():
    print("=" * 60)
    print("  SCENARIO A: VANISHING GRADIENTS")
    print("  Cause: Sigmoid activation — max derivative = 0.25")
    print("  Each layer multiplies gradient by ≤ 0.25")
    print("=" * 60 + "\n")

    torch.manual_seed(42)
    X = torch.randn(32, 16)
    y = torch.randn(32, 1)

    for depth in [5, 10, 20]:
        print(f"── Sigmoid, {depth} hidden layers ──\n")
        torch.manual_seed(42)
        model = make_deep_net(depth, 16, activation="sigmoid", weight_scale=0.5)
        stats = measure_all(model, X, y)
        print_flow(stats, f"sigmoid-{depth}")

    # Show the math
    print("── Why it vanishes ──")
    print("  σ'(z) = σ(z)(1 - σ(z))  →  max = 0.25 at z=0")
    print(f"   5 layers: 0.25^5  = {0.25**5:.2e}")
    print(f"  10 layers: 0.25^10 = {0.25**10:.2e}")
    print(f"  20 layers: 0.25^20 = {0.25**20:.2e}")
    print()


# ──────────────────────────────────────────────
# 5. Scenario B: EXPLODING gradients
# ──────────────────────────────────────────────

def demo_exploding():
    print("=" * 60)
    print("  SCENARIO B: EXPLODING GRADIENTS")
    print("  Cause: Large weight initialization (std=2.0)")
    print("  Each layer multiplies gradient by >> 1")
    print("=" * 60 + "\n")

    torch.manual_seed(42)
    X = torch.randn(32, 16)
    y = torch.randn(32, 1)

    for scale in [0.5, 1.0, 2.0]:
        print(f"── ReLU, 10 layers, weight_scale={scale} ──\n")
        torch.manual_seed(42)
        model = make_deep_net(10, 16, activation="relu", weight_scale=scale)
        stats = measure_all(model, X, y)

        # Check for NaN/Inf
        has_nan = any(s["grad_mean"] != s["grad_mean"] for s in stats["grad_stats"])
        has_inf = any(s["grad_mean"] == float("inf") for s in stats["grad_stats"])

        if has_nan or has_inf:
            print("  ⚠ Gradients are NaN or Inf — completely exploded!\n")
        else:
            print_flow(stats, f"relu-{scale}")

    print("── Why it explodes ──")
    print("  With weight_scale=2.0 and ReLU:")
    print("  Each layer multiplies activations by ~scale * √(width)")
    print("  After N layers: values grow as (scale * √width)^N")
    print(f"  10 layers, scale=2, width=16: ~{(2*16**0.5)**10:.2e}")
    print()


# ──────────────────────────────────────────────
# 6. Side-by-side training: the real impact
# ──────────────────────────────────────────────

def training_impact():
    print("=" * 60)
    print("  TRAINING IMPACT: 10 layers, 200 epochs")
    print("=" * 60 + "\n")

    torch.manual_seed(42)
    X = torch.randn(64, 16)
    y = torch.randn(64, 1)

    configs = [
        ("Vanishing (sigmoid)",       "sigmoid", 0.5,  0.01),
        ("Exploding (relu, big w)",   "relu",    1.5,  0.0001),
        ("Healthy   (relu, He init)", "relu",    None, 0.01),
    ]

    for label, act, scale, lr in configs:
        torch.manual_seed(42)

        if scale is not None:
            model = make_deep_net(10, 16, activation=act, weight_scale=scale)
        else:
            # Proper He initialization
            model = make_deep_net(10, 16, activation="relu", weight_scale=0.01)
            for m in model:
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")

        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
        criterion = nn.MSELoss()

        losses = []
        for epoch in range(1, 201):
            pred = model(X)
            loss = criterion(pred, y)

            # Catch NaN
            if torch.isnan(loss):
                losses.append(float("nan"))
                break

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping for exploding case (common fix)
            if "Exploding" in label:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            losses.append(loss.item())

        e50  = losses[min(49, len(losses)-1)]
        e100 = losses[min(99, len(losses)-1)]
        e200 = losses[min(199, len(losses)-1)]
        print(f"  {label:<30} loss: {e50:>10.4f} → {e100:>10.4f} → {e200:>10.4f}")

    print()


# ──────────────────────────────────────────────
# 7. Fixes demonstration
# ──────────────────────────────────────────────

def demo_fixes():
    print("=" * 60)
    print("  FIXES IN ACTION")
    print("=" * 60 + "\n")

    torch.manual_seed(42)
    X = torch.randn(32, 16)
    y = torch.randn(32, 1)

    # Fix 1: Better activation
    print("── Fix 1: Replace Sigmoid → ReLU ──\n")
    for act in ["sigmoid", "relu"]:
        torch.manual_seed(42)
        model = make_deep_net(10, 16, activation=act, weight_scale=0.5)
        stats = measure_all(model, X, y)
        ratio = stats["grad_stats"][0]["grad_mean"] / (stats["grad_stats"][-1]["grad_mean"] + 1e-30)
        status = "VANISHING" if ratio < 1e-3 else "EXPLODING" if ratio > 1e3 else "HEALTHY"
        print(f"  {act:>8}: gradient ratio = {ratio:.2e}  [{status}]")
    print()

    # Fix 2: Gradient clipping (for exploding)
    print("── Fix 2: Gradient clipping ──\n")
    torch.manual_seed(42)
    model = make_deep_net(10, 16, activation="relu", weight_scale=1.5)
    model.zero_grad()
    loss = nn.MSELoss()(model(X), y)
    loss.backward()

    # Before clipping
    max_before = max(p.grad.abs().max().item()
                     for p in model.parameters() if p.grad is not None)

    # Clip
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # After clipping
    max_after = max(p.grad.abs().max().item()
                    for p in model.parameters() if p.grad is not None)

    print(f"  Before clipping: max |grad| = {max_before:.2e}")
    print(f"  After  clipping: max |grad| = {max_after:.2e}")
    print(f"  Clipped by {max_before/max_after:.0f}×\n")

    # Fix 3: Batch normalization
    print("── Fix 3: Batch normalization ──\n")
    torch.manual_seed(42)
    layers_bn = []
    for _ in range(10):
        layers_bn.append(nn.Linear(16, 16))
        layers_bn.append(nn.BatchNorm1d(16))   # ← normalize between layers
        layers_bn.append(nn.Sigmoid())
    layers_bn.append(nn.Linear(16, 1))
    model_bn = nn.Sequential(*layers_bn)

    model_bn.zero_grad()
    loss = nn.MSELoss()(model_bn(X), y)
    loss.backward()

    grads_bn = [m.weight.grad.abs().mean().item()
                for m in model_bn if isinstance(m, nn.Linear)]
    ratio_bn = grads_bn[0] / (grads_bn[-1] + 1e-30)
    print(f"  Sigmoid + BatchNorm (10 layers):")
    print(f"  Gradient ratio: {ratio_bn:.2e}  ← MUCH better than plain sigmoid")
    print()


# ──────────────────────────────────────────────
# 8. Run everything
# ──────────────────────────────────────────────

if __name__ == "__main__":
    demo_vanishing()
    demo_exploding()
    training_impact()
    demo_fixes()

    print("── Summary ──")
    print("  VANISHING              │  EXPLODING")
    print("  ───────────────────────┼───────────────────────")
    print("  Sigmoid / Tanh         │  Large weight init")
    print("  Gradients → 0          │  Gradients → ∞ / NaN")
    print("  Early layers frozen    │  Weights oscillate wildly")
    print("  ───────────────────────┼───────────────────────")
    print("  Fix: ReLU / GELU       │  Fix: gradient clipping")
    print("  Fix: skip connections  │  Fix: proper init (He)")
    print("  Fix: batch norm        │  Fix: lower learning rate")
    print("  Fix: He / Xavier init  │  Fix: batch norm")

  SCENARIO A: VANISHING GRADIENTS
  Cause: Sigmoid activation — max derivative = 0.25
  Each layer multiplies gradient by ≤ 0.25

── Sigmoid, 5 hidden layers ──

  Gradients (weight.grad):
  Layer    Mean |grad|      Bar
  0        7.65e-04         ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  1        5.74e-03         ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  2        1.78e-02         ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  3        3.58e-02         ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  4        8.16e-02         █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  5        1.62e+00         ███████████████████████████████████
  Ratio (layer 0 / layer 5): 4.73e-04
  ⚠ VANISHING

  Activations (after each activation fn):
  Layer    Mean |act|     Std           
  0        0.5064         0.3047        
  1        0.4635         0.2240        
  2        0.4311         0.1765        
  3        0.4546         0.1835        
  4        0.5098         0.2194        

── Sigmoid, 10 hidden layers ──

  Gradients (weight.